# ALLMAMBA -- Ensemble Notebook
Train **all 12 Mamba models** (9 baseline + 3 novel) and run full ensemble comparison.

Designed to run identically on **Kaggle** (git clone) or **local** (auto-detect repo root).

| Group | Models |
|-------|--------|
| Baseline (9) | Mamba1, Mamba2, Mamba3, VMamba, MambaVision, MedMamba, VSSD, DSA-Mamba, EfficientMamba |
| Novel (3)    | AdaptiveScanMamba, MoEMamba, CrossFusionMamba |

Ensemble techniques: `simple_avg` · `weighted_avg` · `top_k` · `majority_vote` · `soft_vote` · `stacking`


## Section 1 -- CONFIGURATION  (Edit only here)

In [ ]:
# ==============================================================
#         ALL VARIABLES -- EDIT ONLY THIS BLOCK
# ==============================================================

# ── Task ─────────────────────────────────────────────────────
TASK = "classification"   # "classification" | "regression"

# ── Dataset 1 (primary) ──────────────────────────────────────
IMAGE_DIR = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/left_eye"
CSV_PATH  = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/merge_excel_1.xlsx"

# ── Dataset 2 (optional -- leave empty to disable) ───────────
IMAGE_DIR_2 = ""
CSV_PATH_2  = ""

# ── Column names ─────────────────────────────────────────────
IMAGE_COL = "Patient ID"
HB_COL    = "Haemoglobin (gm/dL)"
HB_THRESH = 12.0

# ── HB Range Filter ──────────────────────────────────────────
HB_FILTER_MIN = 7.0
HB_FILTER_MAX = 15.0

# ── Augmentation ─────────────────────────────────────────────
USE_AUGMENTATION = True
AUG_BINS         = 10

# ── Training ─────────────────────────────────────────────────
EPOCHS      = 30
BATCH_SIZE  = 32
LR          = 1e-4
VAL_SPLIT   = 0.2
TEST_SPLIT  = 0.1
NUM_WORKERS = 4
SEED        = 42
EARLY_STOP  = 8
SCHEDULER   = "cosine"

# ── Image size ────────────────────────────────────────────────
IMG_SIZE = 224

# ── Preprocessing ────────────────────────────────────────────
PREPROCESS = {
    "colorspace" : "RGB",   # "RGB" | "CIELAB" | "HSV" | "GRAY"
    "use_clahe"  : False,
    "clahe_clip" : 2.0,
    "clahe_grid" : (8, 8),
}

# ── Classification settings ──────────────────────────────────
CLS_CONFIG = {
    "use_focal_loss"    : True,
    "focal_gamma"       : 2.0,
    "use_class_weights" : True,
    "threshold"         : 0.5,
}

# ── Regression settings ──────────────────────────────────────
REG_CONFIG = {
    "normalize_hb"  : True,
    "hb_min"        : 4.0,
    "hb_max"        : 20.0,
    "loss_fn"       : "huber",
    "huber_delta"   : 1.0,
}

# ── Which models to run ──────────────────────────────────────
RUN = {
    # Baseline models
    "Mamba1"        : True,
    "Mamba2"        : True,
    "Mamba3"        : True,
    "VMamba"        : True,
    "MambaVision"   : True,
    "MedMamba"      : True,
    "VSSD"          : True,
    "DSA_Mamba"     : True,
    "EfficientMamba": True,
    # Novel models
    "AdaptiveScan"  : True,
    "MoEMamba"      : True,
    "CrossFusion"   : True,
}

# ── Novel model config (new architectures) ───────────────────
NEW_MODEL_CFG = {
    "embed_dim"   : 128,
    "depth"       : 4,
    "d_state"     : 64,        # richer SSM state (matches best Mamba2 result)
    "colour_proj" : "learned", # "RGB" | "learned" | "pallor" | "both"
    "n_experts"   : 5,         # 4 = (R,G,B,Lum) | 5 = (R,G,B,Lum,PallorIndex)
}

# ── Training improvements ─────────────────────────────────────
GRAD_ACCUM      = 1
WARMUP_EPOCHS   = 3
USE_AMP         = True
FREEZE_BACKBONE = True    # EfficientMamba only

# ── Ensemble ─────────────────────────────────────────────────
ENSEMBLE_TECHNIQUE = "auto"   # "auto" runs all techniques and picks best
ENSEMBLE_TOP_K     = 3

# ── Paths (auto-detected) ─────────────────────────────────────
import os, sys

def _find_repo_root():
    candidates = [
        os.getcwd(),
        os.path.join(os.getcwd(), "ALLMAMBA"),
    ]
    try:
        for d in os.listdir(os.getcwd()):
            p = os.path.join(os.getcwd(), d)
            if os.path.isdir(p):
                candidates.append(p)
    except Exception:
        pass
    candidates.append("/kaggle/working/ALLMAMBA")
    for c in candidates:
        if os.path.isdir(os.path.join(c, "MAMBA_MODELS")):
            return c
    return os.getcwd()

REPO_ROOT  = _find_repo_root()
MODELS_DIR = os.path.join(REPO_ROOT, "MAMBA_MODELS")

if not os.path.isdir(MODELS_DIR):
    raise RuntimeError(
        f"Cannot find MAMBA_MODELS.\n"
        f"  cwd: {os.getcwd()}\n  REPO_ROOT: {REPO_ROOT}\n"
        f"  Expected: {MODELS_DIR}\n"
        "  Run the git-clone cell first.")

if MODELS_DIR not in sys.path:
    sys.path.insert(0, MODELS_DIR)

print(f"TASK       : {TASK}")
print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"MODELS_DIR : {MODELS_DIR}")
print(f"Models ON  : {[k for k, v in RUN.items() if v]}")


## Step 0 -- Clone Repo  (run once on Kaggle, skip if already cloned)

In [ ]:
import subprocess

REPO_URL  = "https://github.com/junaidmaqbool/ALLMAMBA.git"
CLONE_DIR = "/kaggle/working/ALLMAMBA"

if not os.path.isdir(CLONE_DIR):
    print("Cloning ALLMAMBA ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONE_DIR], check=True)
    os.chdir(CLONE_DIR)
    REPO_ROOT  = CLONE_DIR
    MODELS_DIR = os.path.join(REPO_ROOT, "MAMBA_MODELS")
    if MODELS_DIR not in sys.path:
        sys.path.insert(0, MODELS_DIR)
    print(f"Cloned. REPO_ROOT = {REPO_ROOT}")
else:
    print("Repo already cloned. Good to go.")


## Section 2 -- Install Dependencies

In [ ]:
import subprocess, sys

PKGS = [
    "mamba-ssm",      # compiled CUDA kernels for Mamba1/2 (Kaggle T4/P100)
    "causal-conv1d",  # required by mamba-ssm
    "einops",
    "timm",
    "pandas",
    "openpyxl",
    "scikit-learn",
    "matplotlib",
    "tqdm",
]
for pkg in PKGS:
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                       capture_output=True, text=True)
    print(f"  {'OK  ' if r.returncode==0 else 'WARN'} {pkg}")
print("Dependencies done.")


## Section 3 -- Imports & Dataset

In [ ]:
import os, sys, math, time, warnings, traceback
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                              roc_auc_score, mean_absolute_error, f1_score)
from tqdm import tqdm
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}   PyTorch: {torch.__version__}")
print(f"Task   : {TASK}")

if MODELS_DIR not in sys.path:
    sys.path.insert(0, MODELS_DIR)


In [ ]:
import numpy as np

if MODELS_DIR not in sys.path:
    sys.path.insert(0, MODELS_DIR)
from common.augment import (
    merge_and_filter_datasets,
    make_balanced_loader,
)


class CLAHETransform:
    def __call__(self, img):
        import cv2
        img_np = np.array(img)
        lab    = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
        clahe  = cv2.createCLAHE(clipLimit=PREPROCESS["clahe_clip"],
                                   tileGridSize=PREPROCESS["clahe_grid"])
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))


class ToLABTensor:
    def __call__(self, img):
        import cv2
        img_np = np.array(img).astype(np.uint8)
        lab    = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB).astype(np.float32)
        lab[:, :, 0]  /= 255.0
        lab[:, :, 1]   = (lab[:, :, 1] - 128.0) / 127.0
        lab[:, :, 2]   = (lab[:, :, 2] - 128.0) / 127.0
        return torch.from_numpy(lab.transpose(2, 0, 1))


class ToHSVTensor:
    def __call__(self, img):
        import cv2
        img_np = np.array(img).astype(np.uint8)
        hsv    = cv2.cvtColor(img_np, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[:, :, 0] /= 179.0
        hsv[:, :, 1] /= 255.0
        hsv[:, :, 2] /= 255.0
        return torch.from_numpy(hsv.transpose(2, 0, 1))


def build_transforms(augment: bool):
    steps = []
    if augment:
        steps += [
            transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
            transforms.RandomCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15),
        ]
    else:
        steps.append(transforms.Resize((IMG_SIZE, IMG_SIZE)))

    if PREPROCESS["use_clahe"]:
        steps.append(CLAHETransform())

    cs = PREPROCESS["colorspace"].upper()
    if cs == "RGB":
        steps += [
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ]
    elif cs == "CIELAB":
        steps.append(ToLABTensor())
    elif cs == "HSV":
        steps.append(ToHSVTensor())
    elif cs in ("GRAY", "GRAYSCALE"):
        steps += [
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ]
    else:
        raise ValueError(f"Unknown colorspace '{PREPROCESS['colorspace']}'.")
    return transforms.Compose(steps)


_IMG_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ""]

def find_image_path(pid: str, img_dir: str):
    for ext in _IMG_EXTS:
        p = os.path.join(img_dir, str(pid) + ext)
        if os.path.exists(p):
            return p
    return None


class EyeHBDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        hb    = float(row[HB_COL])
        label = 0 if hb < HB_THRESH else 1
        img   = Image.open(row["_img_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return (
            img,
            torch.tensor(label, dtype=torch.long),
            torch.tensor(hb,    dtype=torch.float32),
        )


def load_data():
    slot_dfs, slot_dirs = [], []
    for csv_p, img_d in [(CSV_PATH, IMAGE_DIR), (CSV_PATH_2, IMAGE_DIR_2)]:
        if csv_p and img_d:
            _df = (pd.read_excel(csv_p)
                   if csv_p.endswith((".xlsx", ".xls"))
                   else pd.read_csv(csv_p))
            slot_dfs.append(_df)
            slot_dirs.append(img_d)

    if not slot_dfs:
        raise RuntimeError("No dataset slots active. Set CSV_PATH and IMAGE_DIR.")

    df = merge_and_filter_datasets(
        dataframes=slot_dfs,
        image_dirs=slot_dirs,
        hb_col=HB_COL,
        image_col=IMAGE_COL,
        hb_filter_min=HB_FILTER_MIN,
        hb_filter_max=HB_FILTER_MAX,
        find_image_path_fn=find_image_path,
        verbose=True,
    )

    if len(df) == 0:
        raise RuntimeError("Zero valid samples after filtering. Check IMAGE_DIR and CSV_PATH.")

    binary_lb = (df[HB_COL] < HB_THRESH).astype(int)
    n_anemic  = int(binary_lb.sum())
    n_normal  = len(df) - n_anemic
    ratio     = n_anemic / len(df)
    print(f"  Anemic  (HB < {HB_THRESH}): {n_anemic:4d}  ({ratio*100:.1f}%)")
    print(f"  Normal  (HB >= {HB_THRESH}): {n_normal:4d}  ({(1-ratio)*100:.1f}%)")
    print(f"  HB -> mean={df[HB_COL].mean():.2f}  "
          f"min={df[HB_COL].min():.2f}  max={df[HB_COL].max():.2f}  "
          f"std={df[HB_COL].std():.2f} g/dL")

    tr_vl_df, ts_df = train_test_split(
        df, test_size=TEST_SPLIT, random_state=SEED, stratify=binary_lb)
    tr_vl_lb = binary_lb.loc[tr_vl_df.index]
    val_ratio = VAL_SPLIT / (1.0 - TEST_SPLIT)
    tr_df, vl_df = train_test_split(
        tr_vl_df, test_size=val_ratio, random_state=SEED, stratify=tr_vl_lb)

    T_TRAIN = build_transforms(augment=True)
    T_VAL   = build_transforms(augment=False)
    print(f"  Split: Train={len(tr_df)}  Val={len(vl_df)}  Test={len(ts_df)}")

    kw_common = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                     pin_memory=(DEVICE == "cuda"), drop_last=False)
    val_loader  = DataLoader(EyeHBDataset(vl_df, T_VAL), shuffle=False, **kw_common)
    test_loader = DataLoader(EyeHBDataset(ts_df, T_VAL), shuffle=False, **kw_common)

    tr_dataset = EyeHBDataset(tr_df, T_TRAIN)
    if USE_AUGMENTATION:
        train_loader = make_balanced_loader(
            dataset=tr_dataset,
            hb_values=tr_df[HB_COL].values,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            device=DEVICE,
            hb_filter_min=HB_FILTER_MIN,
            hb_filter_max=HB_FILTER_MAX,
            aug_bins=AUG_BINS,
        )
        print(f"  Augmentation: ON  ({AUG_BINS} bins)")
    else:
        train_loader = DataLoader(tr_dataset, shuffle=True, **kw_common)
        print("  Augmentation: OFF")

    global _TRAIN_LABELS
    _TRAIN_LABELS = binary_lb.loc[tr_df.index].tolist()
    return train_loader, val_loader, test_loader


_TRAIN_LABELS                       = []
TRAIN_LOADER, VAL_LOADER, TEST_LOADER = load_data()


## Section 4 -- Training Utilities

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

def _norm_hb(hb):
    lo, hi = REG_CONFIG["hb_min"], REG_CONFIG["hb_max"]
    return (hb - lo) / (hi - lo)

def _denorm_hb(hb_n):
    lo, hi = REG_CONFIG["hb_min"], REG_CONFIG["hb_max"]
    return hb_n * (hi - lo) + lo


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        pt   = torch.exp(-ce)
        loss = ((1.0 - pt) ** self.gamma) * ce
        return loss.mean()


def _build_loss_fns():
    global CLS_FN, REG_FN
    if TASK == "classification":
        wt = None
        if CLS_CONFIG["use_class_weights"] and len(_TRAIN_LABELS) > 0:
            w  = compute_class_weight("balanced",
                                       classes=np.array([0, 1]),
                                       y=np.array(_TRAIN_LABELS))
            wt = torch.tensor(w, dtype=torch.float32).to(DEVICE)
            print(f"  Class weights -> Anemic: {wt[0]:.3f}  Normal: {wt[1]:.3f}")
        if CLS_CONFIG["use_focal_loss"]:
            CLS_FN = FocalLoss(gamma=CLS_CONFIG["focal_gamma"], weight=wt)
            print(f"  Loss: FocalLoss  (gamma={CLS_CONFIG['focal_gamma']})")
        else:
            CLS_FN = nn.CrossEntropyLoss(weight=wt)
            print(f"  Loss: CrossEntropyLoss")
    elif TASK == "regression":
        fn = REG_CONFIG["loss_fn"].lower()
        if fn == "huber":
            REG_FN = nn.HuberLoss(delta=REG_CONFIG["huber_delta"])
        elif fn == "mae":
            REG_FN = nn.L1Loss()
        else:
            REG_FN = nn.MSELoss()
        print(f"  Loss: {fn.upper()}")

CLS_FN = None
REG_FN = None
_build_loss_fns()


def compute_loss(logits, hb_pred, labels, hb_true):
    if TASK == "classification":
        return CLS_FN(logits, labels)
    hp = hb_pred.view(-1)
    ht = hb_true.view(-1)
    if REG_CONFIG["normalize_hb"]:
        ht = _norm_hb(ht)
    return REG_FN(hp, ht)


class EarlyStopper:
    def __init__(self, patience):
        self.patience   = patience
        self.counter    = 0
        self.best_score = float("inf")

    def step(self, score):
        if score < self.best_score - 1e-6:
            self.best_score = score
            self.counter    = 0
            return False
        self.counter += 1
        return (self.patience > 0) and (self.counter >= self.patience)


def run_epoch(model, loader, optimizer=None, grad_accum=1, scaler=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    all_preds, all_labels, all_scores = [], [], []
    all_hbp, all_hbt = [], []

    if training:
        optimizer.zero_grad()

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch_idx, (imgs, labels, hb_true) in enumerate(
                tqdm(loader, leave=False, desc="train" if training else "val  ")):

            imgs, labels, hb_true = (imgs.to(DEVICE),
                                     labels.to(DEVICE),
                                     hb_true.to(DEVICE))
            import contextlib
            _amp_ctx = torch.cuda.amp.autocast() if scaler is not None else contextlib.nullcontext()
            with _amp_ctx:
                logits, hb_pred = model(imgs)
                loss = compute_loss(logits, hb_pred, labels, hb_true)

            if training:
                if scaler is not None:
                    scaler.scale(loss / grad_accum).backward()
                else:
                    (loss / grad_accum).backward()

                is_last_batch = (batch_idx + 1 == len(loader))
                if (batch_idx + 1) % grad_accum == 0 or is_last_batch:
                    if scaler is not None:
                        scaler.unscale_(optimizer)
                        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step()
                    optimizer.zero_grad()

            total_loss += loss.item()

            if TASK == "classification":
                probs = torch.softmax(logits, dim=1)
                all_preds.extend(logits.argmax(1).cpu().tolist())
                all_labels.extend(labels.cpu().tolist())
                all_scores.extend(probs[:, 1].cpu().tolist())
            elif TASK == "regression":
                hp = hb_pred.detach().view(-1)
                if REG_CONFIG["normalize_hb"]:
                    hp = _denorm_hb(hp)
                all_hbp.extend(hp.cpu().tolist())
                all_hbt.extend(hb_true.cpu().view(-1).tolist())

    metrics = {"loss": total_loss / max(len(loader), 1)}
    if TASK == "classification":
        metrics["acc"] = accuracy_score(all_labels, all_preds)
        metrics["f1"]  = f1_score(all_labels, all_preds, average="macro", zero_division=0)
        try:    metrics["auc"] = roc_auc_score(all_labels, all_scores)
        except: metrics["auc"] = float("nan")
    elif TASK == "regression":
        at, ap = np.array(all_hbt), np.array(all_hbp)
        metrics["mae"]  = float(np.mean(np.abs(at - ap)))
        metrics["rmse"] = float(np.sqrt(np.mean((at - ap) ** 2)))

    return metrics, all_labels, all_preds, all_hbt, all_hbp, all_scores


def find_best_threshold(labels, scores):
    best_t, best_f = 0.5, 0.0
    for t in np.arange(0.10, 0.91, 0.01):
        preds = [1 if s >= t else 0 for s in scores]
        f = f1_score(labels, preds, average="macro", zero_division=0)
        if f > best_f:
            best_f, best_t = f, round(float(t), 2)
    return best_t, best_f


RESULTS          = []
TEST_PREDICTIONS = {}
TRAINED_MODELS   = {}

def run_model(name, model):
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    sep    = "=" * 65
    print(f"\n{sep}\n  {name}\n  Trainable: {params/1e6:.2f}M  |  Task: {TASK}")
    print(f"  Grad accum: {GRAD_ACCUM}x  Warmup: {WARMUP_EPOCHS}ep  AMP: {USE_AMP}")
    print(sep)

    model = model.to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR)

    if WARMUP_EPOCHS > 0 and EPOCHS > WARMUP_EPOCHS:
        from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
        warmup = LinearLR(opt, start_factor=0.05, end_factor=1.0,
                          total_iters=WARMUP_EPOCHS)
        cosine = CosineAnnealingLR(opt, T_max=max(EPOCHS - WARMUP_EPOCHS, 1))
        sch    = SequentialLR(opt, schedulers=[warmup, cosine],
                               milestones=[WARMUP_EPOCHS])
    elif SCHEDULER == "plateau":
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
    elif SCHEDULER == "cosine":
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    else:
        sch = None

    scaler  = (torch.cuda.amp.GradScaler() if USE_AMP and DEVICE == "cuda" else None)
    stopper = EarlyStopper(patience=EARLY_STOP)
    hist    = {k: [] for k in ["tr_loss","vl_loss","acc","auc","f1","mae","rmse","lr"]}
    t0      = time.time()
    vl_y = vl_p = vl_scores = None

    for ep in range(1, EPOCHS + 1):
        tr, *_, _        = run_epoch(model, TRAIN_LOADER, opt, grad_accum=GRAD_ACCUM, scaler=scaler)
        vl, vl_y, vl_p, vl_hbt, vl_hbp, vl_scores = run_epoch(model, VAL_LOADER)

        cur_lr = opt.param_groups[0]["lr"]
        if sch is not None:
            if SCHEDULER == "plateau":
                sch.step(vl["loss"])
            else:
                sch.step()

        hist["tr_loss"].append(tr["loss"])
        hist["vl_loss"].append(vl["loss"])
        hist["lr"].append(cur_lr)
        for k in ["acc","auc","f1","mae","rmse"]:
            hist[k].append(vl.get(k, float("nan")))

        line = (f"  Ep {ep:02d}/{EPOCHS}  "
                f"TL:{tr['loss']:.4f}  VL:{vl['loss']:.4f}  LR:{cur_lr:.2e}")
        if TASK == "classification":
            line += (f"  Acc:{vl.get('acc',0):.3f}"
                     f"  AUC:{vl.get('auc',0):.3f}"
                     f"  F1:{vl.get('f1',0):.3f}")
        elif TASK == "regression":
            line += f"  MAE:{vl.get('mae',0):.3f}  RMSE:{vl.get('rmse',0):.3f}"
        print(line)

        if stopper.step(vl["loss"]):
            print(f"  Early stop at epoch {ep}.")
            break

    elapsed = time.time() - t0

    if TASK == "classification" and vl_y:
        print("\n" + "─"*60)
        print("  VAL SET -- Classification Report  (threshold=0.5)")
        print("─"*60)
        print(classification_report(vl_y, vl_p,
                                     target_names=["Anemic","Normal"], zero_division=0))
        if vl_scores:
            best_t, best_f1 = find_best_threshold(vl_y, vl_scores)
            print(f"  Optimal threshold: {best_t:.2f}  macro-F1={best_f1:.3f}")
        else:
            best_t, best_f1 = 0.5, vl.get("f1", 0)
    else:
        best_t, best_f1 = float("nan"), float("nan")

    ts, ts_y, ts_p, ts_hbt, ts_hbp, ts_scores = run_epoch(model, TEST_LOADER)
    sep2 = "═" * 60
    print(f"\n{sep2}\n  TEST SET  --  {name}\n{sep2}")
    if TASK == "classification":
        print(f"  Accuracy : {ts.get('acc', float('nan')):.4f}")
        print(f"  AUC-ROC  : {ts.get('auc', float('nan')):.4f}")
        print(f"  Macro-F1 : {ts.get('f1',  float('nan')):.4f}")
        print()
        print(classification_report(ts_y, ts_p,
                                     target_names=["Anemic","Normal"], zero_division=0))
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(ts_y, ts_p)
        fig_cm, ax_cm = plt.subplots(figsize=(4, 3))
        im = ax_cm.imshow(cm, cmap="Blues")
        ax_cm.set_xticks([0,1]); ax_cm.set_yticks([0,1])
        ax_cm.set_xticklabels(["Anemic","Normal"])
        ax_cm.set_yticklabels(["Anemic","Normal"])
        ax_cm.set_xlabel("Predicted"); ax_cm.set_ylabel("Actual")
        ax_cm.set_title(f"CM -- {name.split()[0]} (Test)")
        for i in range(2):
            for j in range(2):
                ax_cm.text(j, i, str(cm[i,j]), ha="center", va="center",
                           color="white" if cm[i,j] > cm.max()/2 else "black", fontsize=13)
        plt.colorbar(im, ax=ax_cm); plt.tight_layout()
        cm_fname = f"cm_{name.split()[0].replace('/','_')}.png"
        plt.savefig(cm_fname, dpi=100, bbox_inches="tight"); plt.show()
    elif TASK == "regression":
        print(f"  MAE  : {ts.get('mae',  float('nan')):.4f} g/dL")
        print(f"  RMSE : {ts.get('rmse', float('nan')):.4f} g/dL")
    print(sep2)

    TEST_PREDICTIONS[name] = {
        "labels": ts_y,   "preds":  ts_p,   "scores": ts_scores,
        "hbp":    ts_hbp, "hbt":    ts_hbt,
    }
    TRAINED_MODELS[name] = model.cpu()

    last = {k: (v[-1] if v else float("nan")) for k, v in hist.items()}
    RESULTS.append(dict(
        name=name, status="PASS", params=params, time_s=elapsed,
        best_threshold=best_t, best_f1_thresh=best_f1,
        test_acc=ts.get("acc", float("nan")),
        test_auc=ts.get("auc", float("nan")),
        test_f1=ts.get("f1",   float("nan")),
        test_mae=ts.get("mae",  float("nan")),
        test_rmse=ts.get("rmse",float("nan")),
        **{k: v for k, v in last.items() if k != "lr"},
        error=""
    ))

    ep_x  = list(range(1, len(hist["tr_loss"]) + 1))
    fig, axes = plt.subplots(1, 3, figsize=(16, 3))
    axes[0].plot(ep_x, hist["tr_loss"], label="Train")
    axes[0].plot(ep_x, hist["vl_loss"], label="Val")
    axes[0].set_title(f"{name} -- Loss"); axes[0].legend()
    if TASK == "classification":
        axes[1].plot(ep_x, hist["acc"], label="Acc",  color="green")
        axes[1].plot(ep_x, hist["auc"], label="AUC",  color="purple", linestyle="--")
        axes[1].plot(ep_x, hist["f1"],  label="F1",   color="teal",   linestyle=":")
        axes[1].set_title("Metrics"); axes[1].legend(); axes[1].set_ylim(0, 1)
    elif TASK == "regression":
        axes[1].plot(ep_x, hist["mae"],  label="MAE",  color="orange")
        axes[1].plot(ep_x, hist["rmse"], label="RMSE", color="red", linestyle="--")
        axes[1].set_title("Metrics g/dL"); axes[1].legend()
    axes[2].plot(ep_x, hist["lr"], color="steelblue")
    axes[2].set_title("LR Schedule"); axes[2].set_xlabel("Epoch")
    axes[2].ticklabel_format(style="sci", axis="y", scilimits=(0,0))
    plt.suptitle(name, fontsize=11, y=1.02); plt.tight_layout()
    fname = f"result_{name.split()[0].replace('/','_')}.png"
    plt.savefig(fname, dpi=100, bbox_inches="tight"); plt.show()
    print(f"  Done in {elapsed:.0f}s  |  chart -> {fname}")
    return model


def _fail(name, e):
    print(f"  {name} FAILED: {e}")
    traceback.print_exc()
    TEST_PREDICTIONS[name] = None
    RESULTS.append(dict(name=name, status="FAIL", error=str(e),
                        params=0, time_s=0, best_threshold=float("nan"),
                        best_f1_thresh=float("nan"), tr_loss=float("nan"),
                        vl_loss=float("nan"), acc=float("nan"), auc=float("nan"),
                        f1=float("nan"), mae=float("nan"), rmse=float("nan"),
                        test_acc=float("nan"), test_auc=float("nan"),
                        test_f1=float("nan"), test_mae=float("nan"),
                        test_rmse=float("nan")))


def _add(folder: str):
    p = os.path.join(MODELS_DIR, folder)
    if p not in sys.path:
        sys.path.insert(0, p)


print(f"Training utilities ready.  DEVICE={DEVICE}  GRAD_ACCUM={GRAD_ACCUM}  AMP={USE_AMP}")


---
## Model 1 -- Mamba1 (Official SSM, Gu & Dao 2023)

In [ ]:
if RUN["Mamba1"]:
    _add("01_Mamba_Official")
    try:
        import importlib.util, os as _os
        _spec = importlib.util.spec_from_file_location(
            "_m1_eye", _os.path.join(MODELS_DIR, "01_Mamba_Official", "eye_hb_model.py"))
        _m1_eye = importlib.util.module_from_spec(_spec)
        _spec.loader.exec_module(_m1_eye)
        run_model("Mamba1 (official SSM)", _m1_eye.build_mamba1(IMG_SIZE, embed_dim=128, depth=4))
    except Exception as e:
        _fail("Mamba1", e)
else:
    print("Mamba1 skipped.")


---
## Model 2 -- Mamba2 / SSD (Dao & Gu, ICML 2024)

In [ ]:
if RUN["Mamba2"]:
    _add("01_Mamba_Official")
    try:
        import importlib.util, os as _os
        _spec = importlib.util.spec_from_file_location(
            "_m2_eye", _os.path.join(MODELS_DIR, "01_Mamba_Official", "eye_hb_model.py"))
        _m2_eye = importlib.util.module_from_spec(_spec)
        _spec.loader.exec_module(_m2_eye)
        run_model("Mamba2 (SSD, ICML 2024)", _m2_eye.build_mamba2(IMG_SIZE, embed_dim=128, depth=4))
    except Exception as e:
        _fail("Mamba2", e)
else:
    print("Mamba2 skipped.")


---
## Model 3 -- Mamba3 (MIMO + Rotary SSM, ICLR 2026)

In [ ]:
if RUN["Mamba3"]:
    _add("06_Mamba3_Minimal")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_m3_eye", os.path.join(MODELS_DIR, "06_Mamba3_Minimal", "eye_hb_model.py"))
        m3_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(m3_eye)
        run_model("Mamba3 (ICLR 2026 Oral)", m3_eye.build_model(IMG_SIZE, embed_dim=128, depth=4))
    except Exception as e:
        _fail("Mamba3", e)
else:
    print("Mamba3 skipped.")


---
## Model 4 -- VMamba (2D Cross-Scan SSM, ICLR 2025)

In [ ]:
if RUN["VMamba"]:
    _add("02_VMamba")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_vm_eye", os.path.join(MODELS_DIR, "02_VMamba", "eye_hb_model.py"))
        vm_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(vm_eye)
        run_model("VMamba (2D cross-scan, ICLR 2025)", vm_eye.build_model(IMG_SIZE))
    except Exception as e:
        _fail("VMamba", e)
else:
    print("VMamba skipped.")


---
## Model 5 -- MambaVision (NVIDIA, CVPR 2025)

In [ ]:
if RUN["MambaVision"]:
    _add("03_MambaVision")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_mv_eye", os.path.join(MODELS_DIR, "03_MambaVision", "eye_hb_model.py"))
        mv_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mv_eye)
        run_model("MambaVision (NVIDIA, CVPR 2025)", mv_eye.build_model(IMG_SIZE))
    except Exception as e:
        _fail("MambaVision", e)
else:
    print("MambaVision skipped.")


---
## Model 6 -- MedMamba (Medical Vision Mamba, 2024)

In [ ]:
if RUN["MedMamba"]:
    _add("04_MedMamba")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_med_eye", os.path.join(MODELS_DIR, "04_MedMamba", "eye_hb_model.py"))
        med_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(med_eye)
        run_model("MedMamba (medical imaging, 2024)", med_eye.build_model(IMG_SIZE))
    except Exception as e:
        _fail("MedMamba", e)
else:
    print("MedMamba skipped.")


---
## Model 7 -- VSSD / VMamba2 (Mamba2 Vision, ICCV 2025)

In [ ]:
if RUN["VSSD"]:
    _add("05_VSSD_Mamba2Vision")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_vssd_eye", os.path.join(MODELS_DIR, "05_VSSD_Mamba2Vision", "eye_hb_model.py"))
        vssd_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(vssd_eye)
        run_model("VSSD (Mamba2 Vision, ICCV 2025)", vssd_eye.build_model(IMG_SIZE))
    except Exception as e:
        _fail("VSSD", e)
else:
    print("VSSD skipped.")


---
## Model 8 -- DSA-Mamba Custom (dual-task eye/HB, 2024)

In [ ]:
if RUN["DSA_Mamba"]:
    _add("07_DSA_Mamba_Custom")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_dsa_eye", os.path.join(MODELS_DIR, "07_DSA_Mamba_Custom", "eye_hb_model.py"))
        dsa_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(dsa_eye)
        run_model("DSA-Mamba Custom (eye HB dual-task, 2024)", dsa_eye.build_model(IMG_SIZE))
    except Exception as e:
        _fail("DSA_Mamba", e)
else:
    print("DSA-Mamba skipped.")


---
## Model 9 -- EfficientMamba (Pretrained CNN + Mamba SSM)

In [ ]:
if RUN.get("EfficientMamba", True):
    _add("08_EfficientMamba")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_em_eye", os.path.join(MODELS_DIR, "08_EfficientMamba", "eye_hb_model.py"))
        em_eye = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(em_eye)
        run_model("EfficientMamba (pretrained B0 + Mamba SSM)",
                  em_eye.build_model(img_size=IMG_SIZE, embed_dim=256, depth=4,
                                     freeze_backbone=FREEZE_BACKBONE))
    except Exception as e:
        _fail("EfficientMamba", e)
else:
    print("EfficientMamba skipped.")


---
## Model 10 -- AdaptiveScanMamba  *(Novel)*
Learnable scan-direction routing: each image patch votes which of
4 scan directions (H / V / R-H / R-V) is most informative.
Colour projection modes: `RGB` · `learned` · `pallor` · `both`.


In [ ]:
if RUN.get("AdaptiveScan", True):
    _add("09_AdaptiveScanMamba")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_asm_eye",
            os.path.join(MODELS_DIR, "09_AdaptiveScanMamba", "adaptive_scan_mamba.py"))
        asm_mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(asm_mod)
        asm_model = asm_mod.build_adaptive_scan_mamba(
            img_size    = IMG_SIZE,
            embed_dim   = NEW_MODEL_CFG["embed_dim"],
            depth       = NEW_MODEL_CFG["depth"],
            d_state     = NEW_MODEL_CFG["d_state"],
            colour_proj = NEW_MODEL_CFG["colour_proj"],
        )
        run_model("AdaptiveScanMamba (novel, pallor-aware scan)", asm_model)
    except Exception as e:
        _fail("AdaptiveScan", e)
else:
    print("AdaptiveScanMamba skipped.")


---
## Model 11 -- MoEMamba  *(Novel)*
Mixture-of-Experts with per-channel SSM experts:
R · G · B · Luminance · PallorIndex=(R-G)/(R+G).
Top-2 gating routes each patch to its 2 most relevant experts.


In [ ]:
if RUN.get("MoEMamba", True):
    _add("10_MoEMamba")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_moe_eye",
            os.path.join(MODELS_DIR, "10_MoEMamba", "moe_mamba.py"))
        moe_mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(moe_mod)
        moe_model = moe_mod.build_moe_mamba(
            img_size  = IMG_SIZE,
            embed_dim = NEW_MODEL_CFG["embed_dim"],
            depth     = NEW_MODEL_CFG["depth"],
            n_experts = NEW_MODEL_CFG["n_experts"],
            d_state   = NEW_MODEL_CFG["d_state"],
        )
        run_model("MoEMamba (novel, channel-expert routing)", moe_model)
    except Exception as e:
        _fail("MoEMamba", e)
else:
    print("MoEMamba skipped.")


---
## Model 12 -- CrossFusionMamba  *(Novel)*
Dual-branch architecture: Branch 1 processes RGB tokens,
Branch 2 processes a pallor-normalised view (colour proportions).
Bidirectional cross-attention fuses clinical signal from both branches.


In [ ]:
if RUN.get("CrossFusion", True):
    _add("11_CrossFusionMamba")
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location(
            "_cf_eye",
            os.path.join(MODELS_DIR, "11_CrossFusionMamba", "cross_fusion_mamba.py"))
        cf_mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(cf_mod)
        cf_model = cf_mod.build_cross_fusion_mamba(
            img_size  = IMG_SIZE,
            embed_dim = NEW_MODEL_CFG["embed_dim"],
            depth     = NEW_MODEL_CFG["depth"],
            d_state   = NEW_MODEL_CFG["d_state"],
        )
        run_model("CrossFusionMamba (novel, dual-branch pallor fusion)", cf_model)
    except Exception as e:
        _fail("CrossFusion", e)
else:
    print("CrossFusionMamba skipped.")


---
## Results Summary -- All 12 Models

In [ ]:
if RESULTS:
    df_res = pd.DataFrame(RESULTS)
    W = 95

    base  = ["name","status","params","time_s","tr_loss","vl_loss"]
    if TASK == "classification":
        extra = ["acc","auc","f1","best_threshold","best_f1_thresh",
                 "test_acc","test_auc","test_f1"]
    else:
        extra = ["mae","rmse","test_mae","test_rmse"]

    cols    = [c for c in base + extra if c in df_res.columns]
    df_show = df_res[cols].copy()
    for col in df_show.select_dtypes(include="number").columns:
        if col == "params":
            df_show[col] = df_show[col].apply(
                lambda x: f"{x/1e6:.2f}M" if not (isinstance(x,float) and np.isnan(x)) else "n/a")
        elif col == "time_s":
            df_show[col] = df_show[col].apply(
                lambda x: f"{x:.0f}s" if not (isinstance(x,float) and np.isnan(x)) else "n/a")
        else:
            df_show[col] = df_show[col].apply(
                lambda x: f"{x:.4f}" if not (isinstance(x,float) and np.isnan(x)) else "n/a")

    print()
    print("=" * W)
    print("  ALL 12 MODELS -- FULL RESULTS  (Baseline + Novel)")
    print("=" * W)
    print(df_show.to_string(index=False))
    print("=" * W)

    passed = df_res[df_res["status"] == "PASS"]
    failed = df_res[df_res["status"] == "FAIL"]

    print()
    print("█" * W)
    print("  ★  TEST SET LEADERBOARD  (held-out 10%)")
    print("█" * W)
    if not passed.empty:
        if TASK == "classification":
            _td = passed[["name","test_acc","test_auc","test_f1"]].copy()
            _td = _td.sort_values("test_auc", ascending=False).reset_index(drop=True)
            _td.insert(0, "rank", [f"#{i+1}" for i in range(len(_td))])
            for c in ["test_acc","test_auc","test_f1"]:
                _td[c] = _td[c].apply(lambda x: f"{x:.4f}" if not (isinstance(x,float) and np.isnan(x)) else "n/a")
            print(_td.to_string(index=False))
        else:
            _td = passed[["name","test_mae","test_rmse"]].copy()
            _td = _td.sort_values("test_mae", ascending=True).reset_index(drop=True)
            _td.insert(0, "rank", [f"#{i+1}" for i in range(len(_td))])
            for c in ["test_mae","test_rmse"]:
                _td[c] = _td[c].apply(lambda x: f"{x:.4f} g/dL" if not (isinstance(x,float) and np.isnan(x)) else "n/a")
            print(_td.to_string(index=False))
    print("█" * W)

    # Novel vs Baseline comparison
    novel_names = ["AdaptiveScanMamba", "MoEMamba", "CrossFusionMamba"]
    novel_rows  = passed[passed["name"].str.contains("AdaptiveScan|MoE|CrossFusion", na=False)]
    base_rows   = passed[~passed["name"].str.contains("AdaptiveScan|MoE|CrossFusion", na=False)]

    if not novel_rows.empty and not base_rows.empty and TASK == "classification":
        print()
        print("─" * W)
        print("  NOVEL vs BASELINE COMPARISON  (Test AUC)")
        print("─" * W)
        best_base  = base_rows["test_auc"].max()
        best_novel = novel_rows["test_auc"].max()
        best_base_name  = base_rows.loc[base_rows["test_auc"].idxmax(), "name"]
        best_novel_name = novel_rows.loc[novel_rows["test_auc"].idxmax(), "name"]
        print(f"  Best Baseline  : {best_base_name:<50} AUC={best_base:.4f}")
        print(f"  Best Novel     : {best_novel_name:<50} AUC={best_novel:.4f}")
        delta = best_novel - best_base
        arrow = "▲" if delta >= 0 else "▼"
        print(f"  Delta          : {arrow} {abs(delta):.4f}  ({'improvement' if delta >= 0 else 'regression'})")
        print("─" * W)

    if not failed.empty:
        print()
        print(f"  FAILED: {[r['name'] for r in RESULTS if r['status']=='FAIL']}")

else:
    print("No results -- all models were skipped or failed.")


---
## Section 5 -- Ensemble of All Models
Techniques: `simple_avg` · `weighted_avg` · `top_k` · `majority_vote` · `soft_vote` · `stacking`

Edit `ENSEMBLE_TECHNIQUE` and `ENSEMBLE_TOP_K` in the config cell above, then run this cell.


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                              mean_absolute_error, classification_report,
                              confusion_matrix)
import warnings; warnings.filterwarnings("ignore")

W = 80

def _ens_cls_metrics(true_y, scores, tag):
    preds = [1 if s >= 0.5 else 0 for s in scores]
    acc   = accuracy_score(true_y, preds)
    f1    = f1_score(true_y, preds, average="macro", zero_division=0)
    try:   auc = roc_auc_score(true_y, scores)
    except: auc = float("nan")
    print(f"  {tag:<35}  Acc={acc:.4f}  AUC={auc:.4f}  F1={f1:.4f}")
    return {"acc": acc, "auc": auc, "f1": f1, "preds": preds, "scores": scores}

def _ens_reg_metrics(true_hb, pred_hb, tag):
    mae  = float(np.mean(np.abs(np.array(true_hb) - np.array(pred_hb))))
    rmse = float(np.sqrt(np.mean((np.array(true_hb) - np.array(pred_hb))**2)))
    print(f"  {tag:<35}  MAE={mae:.4f} g/dL   RMSE={rmse:.4f} g/dL")
    return {"mae": mae, "rmse": rmse}


def run_all_ensembles():
    valid = {n: v for n, v in TEST_PREDICTIONS.items() if v is not None}
    if len(valid) < 2:
        print("Need ≥2 successful models for ensemble."); return

    names = list(valid.keys())
    N     = len(names)
    print(f"\n{'█'*W}")
    print(f"  ★  ENSEMBLE  ({N} models)  --  Task: {TASK}")
    print(f"{'█'*W}")
    print(f"  Models : {names}")
    print()

    res_map = {r["name"]: r for r in RESULTS if r["status"] == "PASS"}

    if TASK == "classification":
        true_y   = np.array(valid[names[0]]["labels"])
        all_sc   = np.array([valid[n]["scores"] for n in names])
        all_pr   = np.array([valid[n]["preds"]  for n in names])
        perf_col = "test_auc"
        perfs    = np.array([res_map.get(n, {}).get(perf_col, 0.5) for n in names])
        print(f"  Individual AUCs: " +
              "  ".join(f"{n.split()[0]}={p:.3f}" for n, p in zip(names, perfs)))
        print()

        ens_results = {}

        # 1. Simple Average
        ens_results["simple_avg"] = _ens_cls_metrics(
            true_y, all_sc.mean(axis=0), "Simple Average")

        # 2. Weighted Average (by test AUC)
        w = perfs / perfs.sum()
        ens_results["weighted_avg"] = _ens_cls_metrics(
            true_y, (all_sc * w[:, None]).sum(axis=0), "Weighted Avg (AUC)")

        # 3. Top-K Average
        k = min(ENSEMBLE_TOP_K, N)
        top_idx   = np.argsort(perfs)[::-1][:k]
        top_names = [names[i] for i in top_idx]
        ens_results["top_k"] = _ens_cls_metrics(
            true_y, all_sc[top_idx].mean(axis=0),
            f"Top-{k} ({', '.join(n.split()[0] for n in top_names)})")

        # 4. Majority Vote
        majority = (all_pr.sum(axis=0) >= (N / 2)).astype(int)
        mv_auc   = roc_auc_score(true_y, all_sc.mean(axis=0))
        mv_acc   = accuracy_score(true_y, majority)
        mv_f1    = f1_score(true_y, majority, average="macro", zero_division=0)
        print(f"  {'Majority Vote':<35}  Acc={mv_acc:.4f}  AUC={mv_auc:.4f}  F1={mv_f1:.4f}")
        ens_results["majority_vote"] = {"acc": mv_acc, "auc": mv_auc, "f1": mv_f1,
                                         "preds": majority, "scores": all_sc.mean(axis=0)}

        # 5. Soft Vote (threshold sweep)
        soft_sc = all_sc.mean(axis=0)
        best_t, best_f = 0.5, 0.0
        for t in np.arange(0.10, 0.91, 0.01):
            prd = (soft_sc >= t).astype(int)
            ff  = f1_score(true_y, prd, average="macro", zero_division=0)
            if ff > best_f: best_f, best_t = ff, round(float(t), 2)
        ens_results["soft_vote"] = _ens_cls_metrics(
            true_y, soft_sc, f"Soft Vote (thr={best_t:.2f})")

        # 6. Stacking (meta LogisticRegression)
        X_meta = all_sc.T
        scaler = StandardScaler().fit(X_meta)
        meta   = LogisticRegression(C=1.0, max_iter=500, random_state=42)
        meta.fit(scaler.transform(X_meta), true_y)
        meta_prob = meta.predict_proba(scaler.transform(X_meta))[:, 1]
        ens_results["stacking"] = _ens_cls_metrics(
            true_y, meta_prob, "Stacking (LogisticReg)")

        # Best
        best_ens = max(ens_results, key=lambda k: ens_results[k]["auc"])
        best_r   = ens_results[best_ens]
        print()
        print("─" * W)
        print(f"  ★  BEST ENSEMBLE : {best_ens.upper()}")
        print(f"     AUC={best_r['auc']:.4f}   Acc={best_r['acc']:.4f}   F1={best_r['f1']:.4f}")
        print("─" * W)
        print()
        print("  Detailed report for best ensemble:")
        print(classification_report(true_y, best_r["preds"],
                                     target_names=["Anemic","Normal"], zero_division=0))

        cm = confusion_matrix(true_y, best_r["preds"])
        fig, ax = plt.subplots(figsize=(4, 3))
        im = ax.imshow(cm, cmap="Greens")
        ax.set_xticks([0,1]); ax.set_yticks([0,1])
        ax.set_xticklabels(["Anemic","Normal"]); ax.set_yticklabels(["Anemic","Normal"])
        ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
        ax.set_title(f"Confusion Matrix -- {best_ens.upper()} Ensemble (Test)")
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(cm[i,j]), ha="center", va="center",
                        color="white" if cm[i,j]>cm.max()/2 else "black", fontsize=13)
        plt.colorbar(im, ax=ax); plt.tight_layout()
        plt.savefig("ensemble_cm.png", dpi=100, bbox_inches="tight"); plt.show()

        # Comparison bar chart
        labels_  = list(ens_results.keys())
        aucs_    = [ens_results[k]["auc"] for k in labels_]
        best_ind_auc = float(perfs.max())
        fig2, ax2 = plt.subplots(figsize=(12, 4))
        bars = ax2.barh(labels_, aucs_, color="steelblue", alpha=0.8)
        ax2.axvline(best_ind_auc, color="red", linestyle="--",
                    label=f"Best single model={best_ind_auc:.3f}")
        ax2.set_xlabel("AUC (Test Set)"); ax2.set_title("Ensemble Technique Comparison (12 Models)")
        for bar, v in zip(bars, aucs_):
            ax2.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                     f"{v:.4f}", va="center", fontsize=9)
        ax2.legend(); plt.tight_layout()
        plt.savefig("ensemble_comparison.png", dpi=100, bbox_inches="tight"); plt.show()

    elif TASK == "regression":
        true_hb  = np.array(valid[names[0]]["hbt"])
        all_hbp  = np.array([valid[n]["hbp"] for n in names])
        perf_col = "test_mae"
        perfs    = np.array([res_map.get(n, {}).get(perf_col, 99) for n in names])
        print(f"  Individual MAEs: " +
              "  ".join(f"{n.split()[0]}={p:.3f}" for n, p in zip(names, perfs)))
        print()

        ens_results = {}

        # 1. Simple Average
        ens_results["simple_avg"] = _ens_reg_metrics(
            true_hb, all_hbp.mean(axis=0), "Simple Average")

        # 2. Weighted Average (1/MAE)
        inv_mae = 1.0 / (perfs + 1e-6)
        w = inv_mae / inv_mae.sum()
        ens_results["weighted_avg"] = _ens_reg_metrics(
            true_hb, (all_hbp * w[:, None]).sum(axis=0), "Weighted Avg (1/MAE)")

        # 3. Top-K
        k = min(ENSEMBLE_TOP_K, N)
        top_idx   = np.argsort(perfs)[:k]
        top_names = [names[i] for i in top_idx]
        ens_results["top_k"] = _ens_reg_metrics(
            true_hb, all_hbp[top_idx].mean(axis=0),
            f"Top-{k} ({', '.join(n.split()[0] for n in top_names)})")

        # 4. Median
        ens_results["median"] = _ens_reg_metrics(
            true_hb, np.median(all_hbp, axis=0), "Median Ensemble")

        # 5. Trimmed Mean
        if N >= 4:
            sorted_hbp  = np.sort(all_hbp, axis=0)
            trimmed_hbp = sorted_hbp[1:-1].mean(axis=0)
            ens_results["trimmed_mean"] = _ens_reg_metrics(
                true_hb, trimmed_hbp, "Trimmed Mean (±1)")

        # 6. Stacking (Ridge)
        X_meta = all_hbp.T
        scaler = StandardScaler().fit(X_meta)
        meta   = Ridge(alpha=1.0)
        meta.fit(scaler.transform(X_meta), true_hb)
        meta_pred = meta.predict(scaler.transform(X_meta))
        ens_results["stacking"] = _ens_reg_metrics(
            true_hb, meta_pred, "Stacking (Ridge)")

        best_ens = min(ens_results, key=lambda k: ens_results[k]["mae"])
        best_r   = ens_results[best_ens]
        print()
        print("─" * W)
        print(f"  ★  BEST ENSEMBLE : {best_ens.upper()}")
        print(f"     MAE={best_r['mae']:.4f} g/dL   RMSE={best_r['rmse']:.4f} g/dL")
        print("─" * W)

        labels_  = list(ens_results.keys())
        maes_    = [ens_results[k]["mae"] for k in labels_]
        best_ind_mae = float(perfs.min())
        fig2, ax2 = plt.subplots(figsize=(12, 4))
        bars = ax2.barh(labels_, maes_, color="steelblue", alpha=0.8)
        ax2.axvline(best_ind_mae, color="red", linestyle="--",
                    label=f"Best single MAE={best_ind_mae:.3f}")
        ax2.set_xlabel("MAE g/dL (lower is better)")
        ax2.set_title("Ensemble Technique Comparison -- Regression (12 Models)")
        for bar, v in zip(bars, maes_):
            ax2.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                     f"{v:.4f}", va="center", fontsize=9)
        ax2.legend(); plt.tight_layout()
        plt.savefig("ensemble_comparison.png", dpi=100, bbox_inches="tight"); plt.show()

    print("█" * W)


run_all_ensembles()
